In [1]:
import requests
import time
import pandas as pd

top10_df = pd.read_csv("top10_entities_v2.csv")
top10_df["entity"] = top10_df["entity"].str.strip()

print(f"Entities to code: {len(top10_df)}")
print()

for cat, grp in top10_df.groupby("category"):
    print(f"{cat}: {list(grp['entity'])}")

Entities to code: 40

Condition: ['infection', 'diabetes', 'pneumonia', 'adenocarcinoma', 'carcinoma', 'fracture', 'thrombosis', 'coronary artery disease', 'hiv', 'hepatitis']
Medication: ['antibiotic', 'steroid', 'heparin', 'insulin', 'prednisone', 'warfarin', 'cyclophosphamide', 'dexamethasone', 'prednisolone', 'doxycycline']
Procedure: ['ct', 'physical examination', 'mri', 'biopsy', 'blood pressure', 'ultrasound', 'x-ray', 'laboratory test', 'echocardiography', 'vital sign']
Symptom: ['pain', 'mass', 'tumor', 'lesion', 'fever', 'swelling', 'tenderness', 'bleeding', 'vomiting', 'nodule']


In [2]:
def lookup_condition(term):
    """NLM ICD-10-CM search — returns first match."""
    try:
        r = requests.get(
            "https://clinicaltables.nlm.nih.gov/api/icd10cm/v3/search",
            params={"terms": term, "maxList": 1, "sf": "code,name"},
            timeout=6
        )
        data = r.json()
        if data[0] > 0 and data[3]:
            return data[3][0][0], data[3][0][1]
    except Exception as e:
        print(f"  API error for '{term}': {e}")
    return None, None


def lookup_medication(term):
    """RxNorm API — exact match then approximate fallback."""
    base = "https://rxnav.nlm.nih.gov/REST"
    try:
        r = requests.get(f"{base}/rxcui.json",
                         params={"name": term, "search": 1}, timeout=6)
        rxcui = r.json().get("idGroup", {}).get("rxnormId", [None])[0]

        if not rxcui:
            r2 = requests.get(f"{base}/approximateTerm.json",
                              params={"term": term, "maxEntries": 1}, timeout=6)
            candidates = r2.json().get("approximateGroup", {}).get("candidate", [])
            if candidates:
                rxcui = candidates[0]["rxcui"]

        if rxcui:
            r3    = requests.get(f"{base}/rxcui/{rxcui}/properties.json", timeout=6)
            props = r3.json().get("properties", {})
            return rxcui, props.get("name", term), props.get("tty", "")

    except Exception as e:
        print(f"  RxNorm error for '{term}': {e}")
    return None, None, None


# First pass — hit the APIs and see what comes back
rows = []
for _, row in top10_df.iterrows():
    entity, category = row["entity"], row["category"]

    if category == "Medication":
        rxcui, name, tty = lookup_medication(entity)
        code, desc = rxcui, name
    else:
        code, desc = lookup_condition(entity)

    rows.append({"category": category, "entity": entity,
                 "code": code, "desc": desc})
    time.sleep(0.25)

df_pass1 = pd.DataFrame(rows)
print(f"Coded: {df_pass1['code'].notna().sum()}/40")
print()
print(df_pass1[["category", "entity", "code", "desc"]].to_string(index=False))

Coded: 33/40

  category                  entity     code                                                                             desc
 Condition               infection   K94.02                                                              Colostomy infection
 Condition                diabetes    E23.2                                                               Diabetes insipidus
 Condition               pneumonia   A01.03                                                                Typhoid pneumonia
 Condition          adenocarcinoma     None                                                                             None
 Condition               carcinoma    C22.0                                                             Liver cell carcinoma
 Condition                fracture M84.350A                          Stress fracture, pelvis, initial encounter for fracture
 Condition              thrombosis      I81                                                           Portal ve

In [3]:
def lookup_condition_ranked(term, n=5):
    """
    Fetch top N ICD-10-CM results, score each one.
    Prefer unspecified/general codes over specific anatomical ones —
    appropriate for generic NLP-extracted terms.
    """
    try:
        r = requests.get(
            "https://clinicaltables.nlm.nih.gov/api/icd10cm/v3/search",
            params={"terms": term, "maxList": n, "sf": "code,name"},
            timeout=6
        )
        data = r.json()
        if data[0] == 0 or not data[3]:
            return None, None

        candidates = [(row[0], row[1]) for row in data[3]]

        def score(code, name):
            s = 0
            name_l = name.lower()
            term_l = term.lower()

            if name_l.startswith(term_l):       s += 100
            if "unspecified" in name_l:         s += 60
            s += (6 - len(code.replace(".", ""))) * 15

            bad = ["scrotal", "colostomy", "typhoid", "q fever",
                   "portal", "herpesviral", "insipidus", "right", "left",
                   "initial encounter", "subsequent", "childbirth",
                   "perianal", "cytomegaloviral", "enterostomy"]
            if any(b in name_l for b in bad):   s -= 150
            return s

        best = max(candidates, key=lambda x: score(x[0], x[1]))
        return best[0], best[1]

    except Exception as e:
        print(f"  ICD-10 error for '{term}': {e}")
    return None, None


def lookup_snomed(term):
    """NLM SNOMED-CT search endpoint."""
    try:
        r = requests.get(
            "https://clinicaltables.nlm.nih.gov/api/snomed/v3/search",
            params={"terms": term, "maxList": 3, "sf": "code,name"},
            timeout=6
        )
        data = r.json()
        if data[0] > 0 and data[3]:
            return data[3][0][0], data[3][0][1]
    except Exception as e:
        print(f"  SNOMED error for '{term}': {e}")
    return None, None


# Second pass — improved scoring + SNOMED for procedures
rows2 = []
for _, row in top10_df.iterrows():
    entity, category = row["entity"], row["category"]

    if category == "Medication":
        rxcui, name, tty = lookup_medication(entity)
        code, desc, system = rxcui, name, "RxNorm"
    elif category == "Procedure":
        code, desc = lookup_snomed(entity)
        system = "SNOMED-CT"
    else:
        code, desc = lookup_condition_ranked(entity)
        system = "ICD-10-CM"

    rows2.append({"category": category, "entity": entity,
                  "code": code, "desc": desc, "system": system})
    time.sleep(0.25)

df_pass2 = pd.DataFrame(rows2)
print(f"Coded: {df_pass2['code'].notna().sum()}/40")
print()
print(df_pass2[["category", "entity", "code", "desc", "system"]].to_string(index=False))

  SNOMED error for 'ct': Expecting value: line 1 column 1 (char 0)
  SNOMED error for 'physical examination': Expecting value: line 1 column 1 (char 0)
  SNOMED error for 'mri': Expecting value: line 1 column 1 (char 0)
  SNOMED error for 'biopsy': Expecting value: line 1 column 1 (char 0)
  SNOMED error for 'blood pressure': Expecting value: line 1 column 1 (char 0)
  SNOMED error for 'ultrasound': Expecting value: line 1 column 1 (char 0)
  SNOMED error for 'x-ray': Expecting value: line 1 column 1 (char 0)
  SNOMED error for 'laboratory test': Expecting value: line 1 column 1 (char 0)
  SNOMED error for 'echocardiography': Expecting value: line 1 column 1 (char 0)
  SNOMED error for 'vital sign': Expecting value: line 1 column 1 (char 0)
Coded: 29/40

  category                  entity     code                                                                                     desc    system
 Condition               infection   K94.22                                                 

In [4]:
import sys
!{sys.executable} -m pip install simple-icd-10-cm -q


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: /Users/santhosh/tf310-env/bin/python -m pip install --upgrade pip


In [ ]:
import simple_icd_10_cm as cm

# Hand-picked ICD-10-CM codes 
# Using parent-level 'unspecified' codes on purpose here —
# NLP gives us generic terms like 'infection', not enough context for specifics.
MANUAL_ICD = {
    # Conditions
    "infection":               "B99",
    "diabetes":                "E11",
    "pneumonia":               "J18.9",
    "adenocarcinoma":          "C80.1",
    "carcinoma":               "C80.1",
    "fracture":                "M84.40",
    "thrombosis":              "I82.90",
    "coronary artery disease": "I25.10",
    "hiv":                     "B20",
    "hepatitis":               "B19.9",
    # Symptoms
    "pain":                    "R52",
    "mass":                    "R22.9",
    "tumor":                   "D49.9",
    "lesion":                  "R68.89",
    "fever":                   "R50.9",
    "swelling":                "R60.9",
    "tenderness":              "R68.89",
    "bleeding":                "R58",
    "vomiting":                "R11.10",
    "nodule":                  "R22.9",
}

# SNOMED-CT procedure codes 
# Looked these up manually at browser.ihtsdotools.org
# Went with SNOMED-CT over ICD-10-PCS since PCS is really for inpatient billing
SNOMED_PROCEDURES = {
    "ct":                   ("77477000",  "Computerized axial tomography"),
    "physical examination": ("5880005",   "Physical examination procedure"),
    "mri":                  ("113091000", "Magnetic resonance imaging"),
    "biopsy":               ("86273004",  "Biopsy"),
    "blood pressure":       ("75367002",  "Blood pressure taking"),
    "ultrasound":           ("16310003",  "Diagnostic ultrasonography"),
    "x-ray":                ("363680008", "Radiographic imaging procedure"),
    "laboratory test":      ("15220000",  "Laboratory test"),
    "echocardiography":     ("40701008",  "Echocardiography"),
    "vital sign":           ("46680005",  "Vital signs measurement"),
}

# ATC codes for drug classes 
# RxNorm only handles individual drug ingredients, not classes like 'antibiotic'.
# ATC (WHO classification) is the right system for these.
ATC_CLASSES = {
    "antibiotic": ("J01",   "Antibacterials for systemic use",  "ATC"),
    "steroid":    ("H02AB", "Glucocorticoids",                  "ATC"),
}

# Double-check every ICD code against the actual CDC April 2025 release
print("Validating ICD-10-CM codes — CDC April 2025 release")
print("=" * 65)
for entity, code_val in MANUAL_ICD.items():
    desc     = cm.get_description(code_val)
    billable = cm.is_leaf(code_val)
    depth    = len(cm.get_ancestors(code_val))
    print(f" {entity:28s}  {code_val:8s}  depth={depth}  billable={billable}")
    print(f"     {desc}")

print()
print("All codes valid against CDC April 2025 ICD-10-CM ✓")
print("Non-billable codes are intentional — parent-level codes")
print("appropriate for generic NLP-extracted terms")

Validating ICD-10-CM codes — CDC April 2025 release
 infection                     B99       depth=2  billable=False
     Other and unspecified infectious diseases
 diabetes                      E11       depth=2  billable=False
     Type 2 diabetes mellitus
 pneumonia                     J18.9     depth=3  billable=True
     Pneumonia, unspecified organism
 adenocarcinoma                C80.1     depth=3  billable=True
     Malignant (primary) neoplasm, unspecified
 carcinoma                     C80.1     depth=3  billable=True
     Malignant (primary) neoplasm, unspecified
 fracture                      M84.40    depth=4  billable=False
     Pathological fracture, unspecified site
 thrombosis                    I82.90    depth=4  billable=True
     Acute embolism and thrombosis of unspecified vein
 coronary artery disease       I25.10    depth=4  billable=True
     Atherosclerotic heart disease of native coronary artery without angina pectoris
 hiv                           B20      

In [ ]:
# Put it all together, API results + manual corrections 

final_rows = []

for _, row in top10_df.iterrows():
    entity, category = row["entity"], row["category"]

    # Conditions and Symptoms — use our hand-verified ICD codes
    if category in ("Condition", "Symptom") and entity in MANUAL_ICD:
        code_val = MANUAL_ICD[entity]
        desc     = cm.get_description(code_val)
        system   = "ICD-10-CM"
        method   = "manual-verified (CDC April 2025)"

    # Anything not in our manual table — fall back to the ranked API
    elif category in ("Condition", "Symptom"):
        code_val, desc = lookup_condition_ranked(entity)
        system   = "ICD-10-CM"
        method   = "api-ranked"
        time.sleep(0.2)

    # Drug classes get ATC codes (RxNorm doesn't do classes)
    elif category == "Medication" and entity in ATC_CLASSES:
        code_val, desc, system = ATC_CLASSES[entity]
        method = "manual-atc (drug class)"

    # Individual drugs — RxNorm API handles these fine
    elif category == "Medication":
        rxcui, name, tty = lookup_medication(entity)
        code_val = rxcui
        desc     = name
        system   = "RxNorm"
        method   = f"api-rxnorm (tty={tty})"
        time.sleep(0.2)

    # Procedures — grab from our SNOMED lookup table
    elif category == "Procedure":
        if entity in SNOMED_PROCEDURES:
            code_val, desc = SNOMED_PROCEDURES[entity]
            system   = "SNOMED-CT"
            method   = "manual-verified (browser.ihtsdotools.org)"
        else:
            code_val, desc = None, None
            system   = "SNOMED-CT"
            method   = "not-found"

    else:
        code_val, desc, system, method = None, None, "Unknown", "not-found"

    final_rows.append({
        "category":       category,
        "rank":           row["rank"],
        "entity":         entity,
        "record_count":   row["record_count"],
        "avg_confidence": round(row["avg_confidence"], 3),
        "code":           code_val,
        "code_desc":      desc,
        "code_system":    system,
        "lookup_method":  method,
    })

coded_df = pd.DataFrame(final_rows)

# How'd we do?
print(f"Final coverage: {coded_df['code'].notna().sum()}/{len(coded_df)}")
print()
print("By lookup method:")
print(coded_df["lookup_method"].value_counts().to_string())
print()
coded_df.to_csv("top10_coded.csv", index=False)
print("Saved → top10_coded.csv")

Final coverage: 40/40

By lookup method:
lookup_method
manual-verified (CDC April 2025)             20
manual-verified (browser.ihtsdotools.org)    10
api-rxnorm (tty=PIN)                          5
api-rxnorm (tty=IN)                           3
manual-atc (drug class)                       2

Saved → top10_coded.csv


In [8]:
# Final table — this is what the dashboard reads
print(coded_df[[
    "category", "rank", "entity", "record_count",
    "code", "code_desc", "code_system", "lookup_method"
]].sort_values(["category", "rank"]).to_string(index=False))

  category  rank                  entity  record_count      code                                                                       code_desc code_system                             lookup_method
 Condition     1               infection            27       B99                                       Other and unspecified infectious diseases   ICD-10-CM          manual-verified (CDC April 2025)
 Condition     2                diabetes            24       E11                                                        Type 2 diabetes mellitus   ICD-10-CM          manual-verified (CDC April 2025)
 Condition     3               pneumonia            23     J18.9                                                 Pneumonia, unspecified organism   ICD-10-CM          manual-verified (CDC April 2025)
 Condition     4          adenocarcinoma            19     C80.1                                       Malignant (primary) neoplasm, unspecified   ICD-10-CM          manual-verified (CDC April 2025)
 Cond